# Lab 9: Is 2 percent still enough?

It is mid-July 2026. You are first-year staff on the audit of **Summit Gear Co.** for the
fiscal year ended **June 30, 2026**. Summit Gear is privately held. Management books its
allowance for doubtful accounts at a flat **2 percent of gross receivables**, under a
policy memo from December 2024. Your senior asked you to build an independent
expectation of the allowance from Summit Gear's own aging history and tell her whether
2 percent holds.

The lab has five parts, one per step of the model:

- **Part 1, describe.** Age the June 30 balance and tie it to the general ledger.
- **Part 2, loss rates.** How much of each aging bucket was later written off, and how
  last year's allowance compared with what actually came.
- **Part 3, the expectation.** Your allowance, the booked one, and which segment explains
  the difference.
- **Part 4, sensitivity.** How far each rate moves your answer, and what it would take
  for 2 percent to be enough.
- **Part 5, the note.** Four to eight sentences to your senior, written in the Canvas quiz.

Ten functions are graded. Plan on two to three hours.

> **Keep the reference page open beside this notebook:** [Lab 9 reference page](https://seanmccaman.com/acctg6155/2026-fall/week-9/interactive/lab_09_reference.html).
> It lists every function's specification and every rule in one place.

> **Before anything else:** in Colab, choose **File > Save a copy in Drive** and work in
> your copy. The copy that opens from the link does not save your work.

> **Start here.** Run **Runtime > Run all** now. Nothing will be green yet. The **Final
> check** at the bottom lists what is still to do and names the section to fix.

> **Write each function yourself first.** If you ask an AI for help, give it the rules in
> the next section along with your question. Put the maturity rule in your prompt in
> particular: without it, an AI will usually pool every snapshot in the file.

## The standard: what a complete function looks like

Every function is graded for **completeness**. Here is one that is complete:

```python
def amount_after_discount(amount, discount_rate):
    """Return the amount after an early-payment discount (0.02 means 2%)."""
    discounted = amount * (1 - discount_rate)
    return discounted
```

Each function is graded on three things:
- it is **attempted**: the stub's `return None` is replaced with a real body, and the name
  in the `def` line is kept exactly;
- it has a **docstring**: one plain sentence under the `def` line saying what it returns;
- it **returns the right value**, on the Final check's example and on a few more cases of
  the same kind (an empty bucket, a write-off dated on the snapshot date, a second window).

**The cells you edit are the ones that say `return None`.** Everything else runs as
written. Each function is graded on its own inputs, so a mistake in Part 2 does not cost
you the Part 3 points.

## Section 0: Setup and the data

Colab is the supported path. `pandas` is already installed there. The data is Summit
Gear's general-ledger extract after recorded write-offs; the audit of this year is the work
you are doing now. It is not Week 3's practice extract, which never wrote anything off. The invoices are the same; the collections and write-offs are not, so
the balances differ from Week 3's.

| Table | One row per |
|---|---|
| `snapshots` | open invoice at each month-end, January 31, 2024 to June 30, 2026 |
| `writeoffs` | invoice written off, with the date and the amount |
| `gl` | general-ledger balance at each quarter-end (AR and the allowance) |
| `activity` | quarter of allowance activity: opening, provision, write-offs, ending |
| `customers` | customer, with its segment |

In [ ]:
# Section 0: run this once.
import pandas as pd

DATA = "https://raw.githubusercontent.com/sean-mccaman/acctg5150-090/main/2026-fall/week-09/data/"
snapshots = pd.read_csv(DATA + "open_items_snapshots.csv", parse_dates=["snapshot_date", "invoice_date", "due_date"])
writeoffs = pd.read_csv(DATA + "writeoffs.csv", parse_dates=["written_off_on"])
gl = pd.read_csv(DATA + "gl_balances.csv", parse_dates=["period_end"])
activity = pd.read_csv(DATA + "allowance_activity.csv", parse_dates=["quarter_end"])
customers = pd.read_csv(DATA + "customers.csv")

print("snapshots", snapshots.shape, "| writeoffs", writeoffs.shape, "| gl", gl.shape)
print("snapshot dates:", snapshots["snapshot_date"].min().date(), "to", snapshots["snapshot_date"].max().date())
snapshots.head()

**Prefer SQL?** The same tables ship as one SQLite file. This cell is optional. Use
`pd.read_sql` with any query you would have written in Week 3; the functions below still
take and return pandas objects.

```python
import sqlite3, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/sean-mccaman/acctg5150-090/main/2026-fall/week-09/summit_gear_fy26.sqlite", "summit_gear_fy26.sqlite")
con = sqlite3.connect("summit_gear_fy26.sqlite")
pd.read_sql("SELECT bucket, SUM(open_amount) FROM open_items_snapshots "
            "WHERE snapshot_date = '2026-06-30' GROUP BY bucket", con)
```

## The rules

These decide the numbers. Every one of them is graded, so read them before you write a
function, and paste them into any AI prompt you use.

1. **Write-off policy.** An invoice's remaining balance is written off at the first
   month-end on which it is 180 or more days past due. That is Summit Gear's policy, not
   the credit-loss rule: the expected loss belongs in the allowance well before the
   write-off, which only removes a balance already reserved. Assume there are no
   receipts on previously written-off invoices in this extract.
2. **Buckets.** Days past due at the snapshot date: `Current` (0 or less), `1-30`,
   `31-60`, `61-90`, `Over 90`. Already in the data as `bucket`.
3. **Later written off.** For an open invoice at a snapshot, the dollars of the **same
   `invoice_id`** written off **after** that snapshot date. A write-off dated on the
   snapshot date itself does not count. In the real ledger a written-off invoice is
   already gone from the snapshot on its write-off date, so this rule shows up in the
   Final check's example ledger rather than in your June 30 numbers.
4. **Loss rate (pooled, dollar-weighted).** For a bucket, over the snapshots in a window:
   total dollars later written off from that bucket's invoices, divided by total open
   dollars in that bucket. Add up first, divide once. Do not average the monthly rates.
   Count every write-off, whatever its `reason`. Some are short-payment deductions,
   which in practice are often a sales-allowance question rather than a credit loss;
   this lab pools them, and your note may say so. Add across every snapshot row: an
   invoice that sits in a bucket at three month-ends counts three times, top and bottom. Do
   not collapse to one row per invoice. The repeats are also why thirteen snapshots are not
   thirteen independent samples; that is a caution about the rate, not a step in the
   calculation.
5. **The lookback window.** This lab uses the 13 month-end snapshots from
   **2024-06-30 through 2025-06-30**, inclusive. Earlier snapshots are the ERP's first
   months, before a full aging existed. Every snapshot in the window has had at least
   twelve months to resolve. Under the write-off policy a current invoice reaches 180
   days past due within about seven months, so some later snapshots have resolved too,
   but **the newest have not**: invoices that will be written off are still sitting there
   open, so they contribute no observed write-off yet. A fixed window that has fully resolved keeps that out.
   This is the maturity rule.
6. **Historical range.** For each bucket, the lowest and highest single-snapshot rate
   among the 13 lookback snapshots. A snapshot with no dollars in a bucket is skipped for
   that bucket. If a bucket has no dollars in any snapshot of the window, its low and high
   are both `0.0`.
7. **The senior's threshold: $25,000.** She set it before anyone computed an expectation. A
   difference larger than this needs investigation.
8. **Rounding.** Dollar results are rounded to cents, and they are checked to the cent, so
   rounding before or after a subtraction both pass. Rates are returned unrounded.

In [ ]:
# The rules as constants. Run this; do not change it.
BUCKETS = ["Current", "1-30", "31-60", "61-90", "Over 90"]
AS_OF = "2026-06-30"                 # this year's balance sheet date
PRIOR = "2025-06-30"                 # last year's
LOOKBACK_START, LOOKBACK_END = "2024-06-30", "2025-06-30"
FIRST_SNAPSHOT = "2024-01-31"
POLICY_RATE = 0.02
THRESHOLD = 25000.00

items_asof = snapshots[snapshots["snapshot_date"] == AS_OF]
items_prior = snapshots[snapshots["snapshot_date"] == PRIOR]
gl_ar = gl[gl["account"].str.startswith("1200")].set_index("period_end")["balance"]
gl_allowance = gl[gl["account"].str.startswith("1210")].set_index("period_end")["balance"]
print("open invoices at", AS_OF, ":", len(items_asof))
print("booked allowance at", AS_OF, ":", f"{gl_allowance[AS_OF]:,.2f}")

In [ ]:
# Management's policy memo, as Summit Gear wrote it.
import urllib.request
print(urllib.request.urlopen(DATA + "allowance_policy_memo.txt").read().decode())

# Part 1: Describe

## Section 1: The aging

`aging_table(items)` takes the open items at one snapshot and returns a pandas Series of
open dollars indexed by the five buckets **in the order of `BUCKETS`**, rounded to cents.
A bucket with no invoices shows `0.0`, not a missing row. In Week 7 you grouped and totaled
by segment; this is the same move by bucket, plus putting the rows in order (`.reindex`).

In [ ]:
def aging_table(items):
    """ """
    return None

In [ ]:
aging = aging_table(items_asof)
aging if aging is not None else print("Write aging_table above, then run this cell again.")

## Section 2: Tie it to the general ledger

The first thing anyone does with an aging is tie its total to the receivables account.
`tie_out(aging, gl_balance)` returns the aging total minus the GL balance, rounded to cents.
Zero means it ties.

In [ ]:
def tie_out(aging, gl_balance):
    """ """
    return None

In [ ]:
if aging is None:
    print("Finish Section 1 first.")
else:
    print("tie-out at", AS_OF, ":", tie_out(aging, gl_ar[AS_OF]))

**The allowance roll-forward (provided, not graded).** Opening allowance, plus the
provision, minus write-offs, equals the ending allowance. The second check ties the
year's write-offs in the roll-forward to the `writeoffs` table. Both should print `True`.

In [ ]:
rf = activity.copy()
print("roll-forward foots every quarter:",
      ((rf["opening"] + rf["provision"] - rf["writeoffs"] - rf["ending"]).abs() < 0.005).all())
fy26 = (writeoffs["written_off_on"] >= "2025-07-01") & (writeoffs["written_off_on"] <= AS_OF)
print("FY2026 write-offs tie to the roll-forward:",
      round(writeoffs.loc[fy26, "amount"].sum(), 2) == round(rf.loc[rf["quarter_end"] >= "2025-07-01", "writeoffs"].sum(), 2))
rf.tail(4)

# Part 2: Loss rates

## Section 3: What was later written off

`later_writeoffs(items, writeoffs)` returns a copy of `items` with one new column,
`written_off_later`: for each row, the dollars of the same `invoice_id` written off
**after** that row's `snapshot_date` (rule 3), or `0.0` if none. It works on any number of
snapshots at once, and one invoice can sit in several snapshots, so `invoice_id` alone does
not identify a row. One way: give each row a temporary row number (0 to `len(items) - 1`),
merge the write-offs on `invoice_id`, keep the pairs where `written_off_on` is after that
row's `snapshot_date`, total by the row number, and attach the totals to a copy of `items`,
filling the rest with `0.0`. Keep every row of `items`, in the same order and with the same
index, even when an invoice has more than one write-off.

In [ ]:
def later_writeoffs(items, writeoffs):
    """ """
    return None

## Section 4: The pooled loss rate

`loss_rates(snapshots, writeoffs, start, end)` keeps the snapshots dated from `start`
through `end` (inclusive), and returns a Series of pooled rates by bucket, in `BUCKETS`
order (rule 4). A bucket with no open dollars in the window gets `0.0`. `start` and `end`
arrive as text like `"2024-06-30"`; `pd.Timestamp(start)` turns text into a date you can
compare with.

In [ ]:
def loss_rates(snapshots, writeoffs, start, end):
    """ """
    return None

**Two windows (provided).** The first column is the lookback window from rule 5. The
second uses every snapshot in the file, which adds the ERP's first months and the newest,
unresolved months. Look at which buckets differ most, then answer the quiz question about
which belongs in the expectation.

In [ ]:
try:
    two_ways = pd.DataFrame({
        "lookback (rule 5)": loss_rates(snapshots, writeoffs, LOOKBACK_START, LOOKBACK_END),
        "every snapshot": loss_rates(snapshots, writeoffs, FIRST_SNAPSHOT, AS_OF),
    })
    print(two_ways.map(lambda x: f"{x:.2%}"))
except Exception as e:
    print("Finish Sections 3 and 4 first. (", type(e).__name__, ":", e, ")")

## Section 5: Look back at last year's estimate

An auditor checks how last year's estimate turned out. `backtest_ratio(items, writeoffs,
booked)` returns the dollars later written off from these open items, divided by
`booked`. Run on last year's June 30 balance, a ratio near 1 means last year's 2 percent
covered about what was actually written off.

In [ ]:
def backtest_ratio(items, writeoffs, booked):
    """ """
    return None

In [ ]:
try:
    r = backtest_ratio(items_prior, writeoffs, gl_allowance[PRIOR])
    print(f"Last year's allowance: {gl_allowance[PRIOR]:,.2f}. Later written off, over that: {r:.3f}")
except Exception as e:
    print("Finish Sections 3 and 5 first. (", type(e).__name__, ":", e, ")")

# Part 3: The expectation

## Section 6: Your expected allowance

`expected_allowance(aging, rates)` returns the open dollars in each bucket times that
bucket's rate, added up, rounded to cents. This is the aging-schedule method, one of the
methods the credit-loss standard (ASC 326) names for trade receivables. What you build is a
preliminary expectation from historical rates. A complete estimate would also ask whether
today's balances in a bucket are riskier than the history, and whether forecasts call for
an adjustment; the extract does not settle those questions, and your note can say what
evidence would.

In [ ]:
def expected_allowance(aging, rates):
    """ """
    return None

In [ ]:
booked = gl_allowance[AS_OF]
try:
    rates = loss_rates(snapshots, writeoffs, LOOKBACK_START, LOOKBACK_END)
    expected = expected_allowance(aging, rates)
    print(f"expected {expected:,.2f} | booked {booked:,.2f} | difference {expected - booked:,.2f} | threshold {THRESHOLD:,.2f}")
except Exception as e:
    rates, expected = None, None
    print("Finish Sections 1, 4 and 6 first. (", type(e).__name__, ":", e, ")")

## Section 7: Which segment explains it

`shortfall_by_segment(items, rates, policy_rate)` returns, for each segment, the
bucket-rate allowance on that segment's open items minus `policy_rate` times its open
dollars. Round to cents and sort largest first. A positive number means the bucket-rate
expectation is above the flat-rate allocation for that segment; a negative one means it is
below. It shows where to investigate.

In [ ]:
def shortfall_by_segment(items, rates, policy_rate):
    """ """
    return None

In [ ]:
# Your shortfall_by_segment on the June 30 open items (provided). The quiz asks about this.
try:
    print(shortfall_by_segment(items_asof, rates, POLICY_RATE).to_string())
except Exception as e:
    print("Finish Sections 4, 6 and 7 first. (", type(e).__name__, ":", e, ")")

**Same season, last year (provided).** Before you call a shift a deterioration, compare
June with June. The table shows the share of each segment's open dollars more than 30 days
past due, last year and this year. The bucket rates already price a move into older
buckets, so the shift is in your expectation. ASC 326 also asks whether today's balances
are riskier than past balances in the same bucket, for example whether an invoice now 61 to
90 days past due is less likely to be paid than the history for that bucket says. This table is where
that question starts; answering it takes more evidence, such as collections after June 30.

In [ ]:
def share_over_30(items):
    past = items[items["days_past_due"] > 30].groupby("segment")["open_amount"].sum()
    return (past / items.groupby("segment")["open_amount"].sum()).fillna(0.0)

(pd.DataFrame({"June 30, 2025": share_over_30(items_prior), "June 30, 2026": share_over_30(items_asof)}) * 100).round(1)

# Part 4: Sensitivity

## Section 8: The historical range

`rate_ranges(snapshots, writeoffs, start, end)` returns a DataFrame indexed by the five
buckets in order, with columns `low` and `high`: the lowest and highest single-snapshot
rate in the window (rule 6), and `0.0` for both when a bucket has no dollars in any
snapshot. A single default moves a small bucket's monthly rate a long way, which is why
Part 2 pools thirteen months.

The [allowance explorer](https://seanmccaman.com/acctg6155/2026-fall/week-9/interactive/allowance_explorer.html) shows this idea on a different date if you want
to see it move before you write it.

In [ ]:
def rate_ranges(snapshots, writeoffs, start, end):
    """ """
    return None

## Section 9: The dollar swing

`sensitivity(aging, rates, ranges)` returns a Series by bucket: the expected allowance
with that bucket's rate at its `high`, minus the expected allowance with it at its `low`,
every other bucket held at `rates`. Rounded to cents. The biggest swing is the input to
watch, and it need not be the bucket with the largest reserve.

In [ ]:
def sensitivity(aging, rates, ranges):
    """ """
    return None

## Section 10: What would make 2 percent enough

`breakeven_rate(aging, rates, booked, bucket)` returns the rate for `bucket` that makes the
expected allowance equal `booked`, with every other bucket held at `rates`. If the other
buckets alone already exceed `booked`, the answer is negative; return it anyway, because a
negative breakeven is an answer: no loss rate at or above zero for that bucket gets you
there. If the bucket has no open dollars, return `None`. Breakeven asks for an exact match;
the last two columns of the table below ask the threshold question instead.

In [ ]:
def breakeven_rate(aging, rates, booked, bucket):
    """ """
    return None

In [ ]:
try:
    ranges = rate_ranges(snapshots, writeoffs, LOOKBACK_START, LOOKBACK_END)
    view = ranges.copy()
    view["pooled"] = rates
    view["swing $"] = sensitivity(aging, rates, ranges)
    view["breakeven"] = [breakeven_rate(aging, rates, booked, b) for b in BUCKETS]
    # One move at a time: with this bucket at its low, others pooled, is the gap inside the threshold?
    view["at low: gap"] = [round(expected - aging[b] * (rates[b] - ranges.loc[b, "low"]) - booked, 2) for b in BUCKETS]
    view["inside threshold?"] = view["at low: gap"].abs() <= THRESHOLD
    print(view.to_string())
except Exception as e:
    print("Finish Sections 1, 4, 6, 8, 9 and 10 first. (", type(e).__name__, ":", e, ")")

# Part 5: The note to your senior

Write this in the **Canvas quiz**, not here. Four to eight sentences, plain text. It is
checked against this list, which is printed on the quiz item too:

1. The first sentence states the conclusion: your expected allowance in dollars and
   whether the booked 2 percent is above or below it.
2. That dollar figure matches your notebook's expected allowance.
3. It compares the difference with the $25,000 investigation threshold and says what
   follow-up that calls for. It treats the expectation as preliminary and names the
   evidence still needed. If your conclusion supports an entry, it gives the account
   debited, the account credited, and the full difference, subject to that follow-up.
4. It names what drove the difference, with dollar figures: the segment with the largest
   positive result from `shortfall_by_segment`, and, across all June 30 receivables, the
   aging bucket with the largest expected reserve minus 2 percent of its open dollars.
   The bucket need not be in that segment.
5. It names the input to watch (the bucket with the largest dollar swing in Section 9,
   which can differ from point 4) and whether moving it to its historical low changes
   your conclusion.
6. It names one thing to ask management for, or one procedure still to do (for example,
   collections after June 30 on the balances driving your difference).

Citations to standards are not required. The lead schedule below has the numbers in the
shape a workpaper would. Use the scratch cell if you want to draft here first.

In [ ]:
# The lead schedule (provided). Run after Parts 1 to 4.
try:
    lead = pd.DataFrame({"open dollars": aging, "loss rate": rates, "expected reserve": (aging * rates).round(2)})
    lead["minus 2% of open"] = (lead["expected reserve"] - POLICY_RATE * lead["open dollars"]).round(2)
    lead.loc["Total"] = [aging.sum(), None, round(float((aging * rates).sum()), 2),
                         round(float((aging * rates).sum() - POLICY_RATE * aging.sum()), 2)]
    print(lead.to_string(float_format=lambda x: f"{x:,.4f}" if x < 1 else f"{x:,.2f}"))
    print(f"\nexpected {expected:,.2f}  booked {booked:,.2f}  difference {expected - booked:,.2f}  threshold {THRESHOLD:,.2f}")
except Exception as e:
    print("Finish Parts 1 to 3 first. (", type(e).__name__, ":", e, ")")

In [ ]:
# Scratch space for your note (not graded; the note goes in Canvas).
note = """

"""

# Final check

Run the cell below. It runs each function on a **small example ledger** typed into the
cell (three month-ends, six invoices), where the right answers can be checked by hand.
The example is built to test edge cases, so it does not follow every company policy. It
shows a check mark per function or a cross and **the section to fix**. It does not use
the real June 30 figures. Grading runs more cases of the same kind.

In [ ]:
# ===== RUN THIS to check your work. You do not edit this cell. =====
import io
from IPython.display import display, HTML

TOY_SNAPSHOTS = """snapshot_date,invoice_id,customer_id,segment,invoice_date,due_date,days_past_due,bucket,open_amount
2025-01-31,T-001,C-9001,Resort,2025-01-10,2025-02-09,-9,Current,1000.00
2025-01-31,T-002,C-9002,Outfitter,2024-12-20,2025-01-04,27,1-30,500.00
2025-01-31,T-003,C-9001,Resort,2024-09-15,2024-10-15,108,Over 90,200.00
2025-02-28,T-001,C-9001,Resort,2025-01-10,2025-02-09,19,1-30,1000.00
2025-02-28,T-004,C-9002,Outfitter,2025-02-20,2025-03-07,-7,Current,800.00
2025-03-31,T-001,C-9001,Resort,2025-01-10,2025-02-09,50,31-60,600.00
2025-03-31,T-004,C-9002,Outfitter,2025-02-20,2025-03-07,24,1-30,800.00
2025-03-31,T-005,C-9003,School District,2025-03-25,2025-04-24,-24,Current,2500.00
2025-01-31,T-007,C-9004,Outdoor Retail,2024-09-20,2024-10-20,103,Over 90,400.00
2025-02-28,T-007,C-9004,Outdoor Retail,2024-09-20,2024-10-20,131,Over 90,400.00"""
TOY_WRITEOFFS = """writeoff_id,invoice_id,customer_id,written_off_on,amount,reason
WO-T1,T-003,C-9001,2025-02-28,200.00,180 days past due
WO-T2,T-001,C-9001,2025-08-31,600.00,180 days past due
WO-T3,T-999,C-9009,2025-03-31,75.00,180 days past due
WO-T4,T-002,C-9002,2025-01-31,500.00,180 days past due"""
toy = pd.read_csv(io.StringIO(TOY_SNAPSHOTS), parse_dates=["snapshot_date", "invoice_date", "due_date"])
toy_wo = pd.read_csv(io.StringIO(TOY_WRITEOFFS), parse_dates=["written_off_on"])
toy_mar = toy[toy["snapshot_date"] == "2025-03-31"]
toy_jan = toy[toy["snapshot_date"] == "2025-01-31"]
WANT = {"aging_mar": {"Current": 2500.0, "1-30": 800.0, "31-60": 600.0, "61-90": 0.0, "Over 90": 0.0}, "tie_out": -50.0, "later_jan": [600.0, 0.0, 200.0, 0.0], "rates": {"Current": 0.13953488372093023, "1-30": 0.2608695652173913, "31-60": 1.0, "61-90": 0.0, "Over 90": 0.2}, "backtest_jan": 2.0, "expected_mar": 1157.53, "shortfall_mar": {"Resort": 588.0, "School District": 298.84, "Outfitter": 192.7}, "ranges": {"Current": [0.0, 0.6], "1-30": [0.0, 0.6], "31-60": [1.0, 1.0], "61-90": [0.0, 0.0], "Over 90": [0.0, 0.3333333333333333]}, "sens_mar": {"Current": 1500.0, "1-30": 480.0, "31-60": 0.0, "61-90": 0.0, "Over 90": 0.0}, "breakeven_mar_3160": -0.429221435793731}


CENT = 0.0101   # dollar results are checked to the cent


def final_check():
    """Run one known example per function and show which pass and where to fix the rest."""

    def near(a, b, tol=1e-6):
        return a is not None and b is not None and abs(float(a) - float(b)) <= tol

    def series_is(s, want, tol=1e-6):
        return hasattr(s, "index") and list(s.index) == list(want) and all(near(s[k], v, tol) for k, v in want.items())

    def t_ranges():
        r = rate_ranges(toy, toy_wo, "2025-01-31", "2025-03-31")
        return (list(r.index) == BUCKETS and all(near(r.loc[b, "low"], WANT["ranges"][b][0])
                and near(r.loc[b, "high"], WANT["ranges"][b][1]) for b in BUCKETS))

    def toy_rates():
        return pd.Series(WANT["rates"])

    def toy_ranges():
        return pd.DataFrame({"low": {b: v[0] for b, v in WANT["ranges"].items()},
                             "high": {b: v[1] for b, v in WANT["ranges"].items()}})

    checks = [
        ("aging_table", "Section 1", lambda: series_is(aging_table(toy_mar), WANT["aging_mar"], CENT)),
        ("tie_out", "Section 2", lambda: near(tie_out(pd.Series(WANT["aging_mar"]), 3950.00), WANT["tie_out"], CENT)),
        ("later_writeoffs", "Section 3", lambda: [round(x, 2) for x in later_writeoffs(toy_jan, toy_wo)["written_off_later"]] == WANT["later_jan"]),
        ("loss_rates", "Section 4", lambda: series_is(loss_rates(toy, toy_wo, "2025-01-31", "2025-03-31"), WANT["rates"])),
        ("backtest_ratio", "Section 5", lambda: near(backtest_ratio(toy_jan, toy_wo, 400.00), WANT["backtest_jan"])),
        ("expected_allowance", "Section 6", lambda: near(expected_allowance(pd.Series(WANT["aging_mar"]), toy_rates()), WANT["expected_mar"], CENT)),
        ("shortfall_by_segment", "Section 7", lambda: series_is(shortfall_by_segment(toy_mar, toy_rates(), 0.02), WANT["shortfall_mar"], CENT)),
        ("rate_ranges", "Section 8", t_ranges),
        ("sensitivity", "Section 9", lambda: series_is(sensitivity(pd.Series(WANT["aging_mar"]), toy_rates(), toy_ranges()), WANT["sens_mar"], CENT)),
        ("breakeven_rate", "Section 10", lambda: near(breakeven_rate(pd.Series(WANT["aging_mar"]), toy_rates(), 300.00, "31-60"), WANT["breakeven_mar_3160"])),
    ]

    rows, passed, todo, no_doc = [], 0, [], []
    for label, where, test in checks:
        try:
            ok = bool(test())
        except Exception:
            ok = False
        fn = globals().get(label)
        has_doc = bool(fn is not None and (fn.__doc__ or "").strip())
        if not has_doc:
            no_doc.append(label)
        doc_note = "" if has_doc else "; its docstring is still empty (1 point)"
        if ok:
            passed += 1
            rows.append("<tr><td style='font-size:18px;color:#176b3a'>&#10003;</td>"
                        "<td style='padding:2px 14px'><code>{}</code></td>"
                        "<td style='color:#176b3a'>passes this example{}</td></tr>".format(label, doc_note))
        else:
            todo.append("{} ({})".format(label, where))
            rows.append("<tr><td style='font-size:18px;color:#BE0000'>&#10007;</td>"
                        "<td style='padding:2px 14px'><code>{}</code></td>"
                        "<td style='color:#BE0000'>does not pass yet, fix in <b>{}</b>{}</td></tr>".format(label, where, doc_note))

    color = "#176b3a" if passed == len(checks) else "#BE0000"
    html = ("<div style=\"font-family:Arial,Helvetica,sans-serif\">"
            "<h3 style='margin:4px 0;color:{}'>Lab 9 check: {} of {} functions pass their example</h3>"
            "<table style='border-collapse:collapse;font-size:14px'>{}</table>"
            "<p style='font-size:12.5px;color:#555;margin-top:8px'>A green row means that one example "
            "returned the right value. A docstring is graded only for being there: one sentence in place of "
            "the empty one. The example ledger has a write-off dated on a snapshot date, a write-off for an "
            "invoice that is not in the snapshots, an invoice in the same bucket at two month-ends, and empty buckets; "
            "grading runs more cases like these.</p></div>"
            ).format(color, passed, len(checks), "".join(rows))
    display(HTML(html))
    print("Lab 9 check: {} of {} functions pass their example.".format(passed, len(checks)))
    print("Docstrings filled in: {} of {}.".format(len(checks) - len(no_doc), len(checks)))
    if todo:
        print("Still to do:", "; ".join(todo))
    if no_doc:
        print("Empty docstrings:", ", ".join(no_doc))


final_check()

## Before you submit

- **Run all cells from the top** (Runtime > Run all) so the notebook carries its outputs.
- **Download it:** File > Download > Download .ipynb, and **rename it to
  `<uID>_lab_09.ipynb`**, with your own uID in place of `<uID>`. Upload it to the Lab 9
  quiz, answer the four short questions, and write your note to the senior there.
- **Keep every `def` line exactly as given.** Grading calls your functions by name.
- **Write a real docstring** in each empty `""" """`: one sentence on what it returns.
- **Read an error from its last line first.** If you are stuck, ask the AI to explain the
  error, not to fix it.